#  Unity Catalog Security (RLS + CLS)

This project implements **Row Level Security (RLS)** and **Column Level Security (CLS)** on the Gold layer tables using **Databricks Unity Catalog**.

The goal is to ensure that different business roles can access the same dataset, but see **only the rows and columns they are authorized to view**.



### Groups used

- **Coffee_finance** → Full access to all stores and all columns
- **coffee_store_1_manager** → Access only Store 1
- **coffee_store_2_manager** → Access only Store 2  
- …
- **coffee_store_10_manager** → Access only Store 10

---

##  Gold tables secured

### RLS applied on

- `coffee.gold.fact_transactions`
- `coffee.gold.fact_transaction_items`
- `coffee.gold.dim_stores`

### CLS applied on

- `coffee.gold.dim_users`

---

##  Row Level Security (RLS)

### Business rule

- Finance team can see **all stores**
- Store managers can see **only their assigned store**

This is implemented using:
- A store access mapping table
- A row filter function using `is_account_group_member()`

---

##  Column Level Security (CLS)

### Business rule

PII fields should be visible only to finance users.

Masked columns:
- `gender`
- `birthdate`

Masking behavior:
- Finance → sees original values
- Store manager → sees masked values (`REDACTED` / `NULL`)

---

##  Testing (Finance vs Store Manager)

Security was validated using **two different Databricks logins**, each mapped to a different UC group.

| Role          | Email                        | Group                    |
|--------------|------------------------------|--------------------------|
| Finance       | projectdilpreet@gmail.com     | Coffee_finance           |
| Store Manager | dilpreetkaur6126@gmail.com    | coffee_store_1_manager   |

The same queries were executed from both accounts to confirm:
- RLS filters store-level rows correctly
- CLS masks PII columns correctly

---

##  Dashboard Security Mode

The dashboard was published using:

**Individual data permission**

This ensures that dashboard queries run using the viewer’s identity, so RLS/CLS rules are enforced automatically.


In [0]:
-- ==========================================================
-- TEST 1: Verify group membership (Finance vs Store Manager)
--
-- Purpose:
-- Confirms the logged-in identity and which UC groups
-- the current user belongs to.
--
-- Expected:
-- Finance login:
--   is_finance = true
--   is_store_1 = false
--
-- Store manager login:
--   is_finance = false
--   is_store_1 = true
-- ==========================================================

SELECT
  current_user() AS user,
  is_account_group_member('Coffee_finance') AS is_finance,
  is_account_group_member('coffee_store_1_manager') AS is_store_1;


In [0]:
-- ==========================================================
-- TEST 2: RLS validation on fact_transactions
--
-- Purpose:
-- Checks whether Row Level Security is correctly restricting
-- access to store-specific transaction rows.
--
-- Expected:
-- Finance login:
--   Returns store_id 1–10
--
-- Store manager login:
--   Returns only their assigned store (ex: store_id = 1)
-- ==========================================================

SELECT store_id, COUNT(*) AS txn_count
FROM coffee.gold.fact_transactions
GROUP BY store_id
ORDER BY store_id;


In [0]:
-- ==========================================================
-- TEST 3: RLS validation on fact_transaction_items
--
-- Purpose:
-- Ensures store managers cannot access transaction items
-- belonging to other stores.
--
-- Expected:
-- Finance login:
--   Returns store_id 1–10
--
-- Store manager login:
--   Returns only store_id = 1
-- ==========================================================

SELECT store_id, COUNT(*) AS item_rows
FROM coffee.gold.fact_transaction_items
GROUP BY store_id
ORDER BY store_id;


In [0]:
-- ==========================================================
-- TEST 4: CLS validation on dim_users
--
-- Purpose:
-- Confirms Column Level Security is masking PII fields
-- for store managers.
--
-- Masked columns:
--   - gender
--   - birthdate
--
-- Expected:
-- Finance login:
--   gender + birthdate visible (real values)
--
-- Store manager login:
--   gender = 'REDACTED'
--   birthdate = NULL
-- ==========================================================

SELECT user_id, gender, birthdate, registered_at
FROM coffee.gold.dim_users
LIMIT 20;
